# AI Lab 6 – Simple Text Search (NLP)
**Name:** Mehrab Asghar  |  **Roll No:** 362

In this lab we build a tiny search system for a campus help desk:
1. Split text into words
2. Cut long text into small chunks
3. Find the best chunk for a question (TF-IDF)
4. Check how good the search is

## 1. Imports

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Splitting text into words
Two ways: `split()` (by spaces) and `regex` (only word characters).

In [ ]:
samples = ["Hello, world!", "I can't open Wi-Fi.", "mail@campus.edu", "Room B12 opens at 9.5", ""]

for text in samples:
    by_space = text.split()
    by_regex = re.findall(r"\w+", text)
    print(repr(text))
    print("  split():", by_space, len(by_space))
    print("  regex  :", by_regex, len(by_regex))

## 3. Our documents (campus help desk)

In [ ]:
docs = {
    "wifi": "To join campus Wi-Fi, choose the network named CampusNet. Log in with your student email. "
            "If the login fails, forget the network and connect again. Still not working? Visit the IT room.",
    "library": "The library is open from 8 am to 8 pm on weekdays. You can borrow three books for two weeks. "
               "Late books cost ten rupees per day. Renew books online before the due date.",
    "hostel": "Hostel rooms are given on a first come first served basis. Apply through the student portal. "
              "Visitors are allowed only until 6 pm. Report broken lights or fans to the warden.",
    "canteen": "The canteen serves breakfast from 8 am and lunch from 12 pm. Pay by cash or student card. "
               "Report any food complaint to the canteen manager.",
}

## 4. Chunking
We cut each document into small pieces of 15 words, with 5 words overlapping so we don't lose meaning at the edges.

In [ ]:
def make_chunks(docs, size=15, overlap=5):
    chunks = []
    for name, text in docs.items():
        words = text.split()
        step = size - overlap
        for i in range(0, len(words), step):
            piece = " ".join(words[i:i + size])
            chunks.append({"doc": name, "text": piece})
            if i + size >= len(words):
                break
    return chunks

chunks = make_chunks(docs)
print("Total chunks:", len(chunks))
print(chunks[0])

## 5. Search with TF-IDF
TF-IDF turns text into numbers. Then we compare the question with every chunk using *cosine similarity* (higher = more similar).

In [ ]:
texts = [c["text"] for c in chunks]
vectorizer = TfidfVectorizer(stop_words="english")
matrix = vectorizer.fit_transform(texts)

def search(question, k=2):
    if not question.strip():
        return []
    q = vectorizer.transform([question])
    scores = cosine_similarity(q, matrix)[0]
    best = scores.argsort()[::-1][:k]
    return [(chunks[i], round(float(scores[i]), 3)) for i in best if scores[i] > 0]

for chunk, score in search("How many books can I borrow?"):
    print(score, chunk["doc"], "->", chunk["text"])

## 6. Check the quality
For each question we know the correct document. **Hit@2** = was the right document in the top 2 results?

In [ ]:
tests = [
    ("How do I connect to the internet?", "wifi"),
    ("What is the fine for a late book?", "library"),
    ("Until what time can visitors stay?", "hostel"),
    ("When does lunch start?", "canteen"),
    ("Who won the cricket match?", None),   # not in our documents
]

correct = 0
answerable = 0
for question, right_doc in tests:
    results = search(question)
    found = [c["doc"] for c, s in results]
    if right_doc is None:
        print("(no answer expected)", question, "->", found or "nothing found")
        continue
    answerable += 1
    ok = right_doc in found
    correct += ok
    print("OK " if ok else "BAD", question, "->", found)

print("Hit@2 =", correct, "/", answerable)

## 7. Mini help-desk bot

In [ ]:
def bot(question):
    results = search(question, k=1)
    if not results:
        return "Sorry, I have no answer. Please contact the help desk."
    chunk, score = results[0]
    return f"{chunk['text']}  [source: {chunk['doc']}]"

for q in ["Where can I renew my book?", "", "volcano facts"]:
    print("Q:", repr(q))
    print("A:", bot(q))
    print()

## Conclusion
- `split()` is quick, `regex` cleans punctuation.
- Small chunks with a little overlap help the search find exact answers.
- TF-IDF works well when the question uses the **same words** as the document. Different words (e.g. "internet" vs "Wi-Fi") need smarter methods like embeddings.